# Task 7 — eVolv2k v4: Event Distribution and Aggregation Design

First Band T characterization task. The eVolv2k v4 catalog (Sigl & Toohey 2024, PANGAEA) covers
~2,400 years of volcanic stratospheric sulfur injection (VSSI) events. It is the smallest and
structurally simplest of the three Band T layers — a good entry point for establishing the
Band T characterization workflow.

**Key variables**: `year_ad`, `vssi_tg` (stratospheric sulfur injection, Tg S),
`vssi_1sig` (uncertainty), `lat` (estimated eruption latitude), `asymmetry`
(DG/(DG+DA): 1.0 = NH only, 0.0 = SH only, 0.5 = symmetric).

**Substantive questions** (from `docs/edop/exploration_bandT.md` Task 7):
1. Frequency and VSSI distribution over time — quiet vs. active periods.
2. For typical query windows (50, 100, 200 yr), how often is the window empty?
   How often does it contain ≥1 major event?
3. Which aggregation summary (count, sum-VSSI, time-since-last) is most useful
   for a window query, or should the API expose all three?
4. Does hemispheric asymmetry warrant location-aware filtering in the API?

**Canonical sanity checks**: Tambora 1815, Samalas 1257, Krakatoa 1883,
Kuwae candidate 1452/1453, Mystery eruption 536 CE.

Data: `data/volcano/evolv2k_v4.csv` (no DB required).
Outputs: `output/edop/explore/07_*.{csv,png}`.
Findings: `logs/exploration_log.md` under Task 7.

In [1]:
# Cell 1 — Imports and load catalog

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

DATA  = Path('/Users/karlg/Documents/Repos/_cedop/data/volcano/evolv2k_v4.csv')
OUT   = Path('/Users/karlg/Documents/Repos/_cedop/output/edop/explore')
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)

# Coerce numeric — some fields may be string 'N/A'
for col in ['vssi_tg', 'vssi_1sig', 'asymmetry', 'lat']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Loaded {len(df)} eruption records')
print(f'Year range: {df.year_ad.min()} – {df.year_ad.max()} CE')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst 5 rows:')
print(df[['year_ad','lat','vssi_tg','vssi_1sig','asymmetry','location']].head().to_string(index=False))

Loaded 256 eruption records
Year range: -491 – 1890 CE
Columns: ['year_ad', 'year_iso', 'month', 'day', 'lat', 'so4_grl', 'so4_ant', 'vssi_tg', 'vssi_1sig', 'asymmetry', 'location', 'tephra', 'reference']

First 5 rows:
 year_ad   lat  vssi_tg  vssi_1sig  asymmetry                        location
    1890 -45.0     0.40       0.20      0.000                             NaN
    1886 -38.0     0.74       0.35      0.000 Okataina-Tarawera (New Zealand)
    1883  -6.0     9.34       1.91      0.629            Krakatao (Indonesia)
    1875  65.0     0.67       0.32      1.000                 Askja (Iceland)
    1873  64.4     1.17       0.43      1.000             Grímsvötn (Iceland)


In [2]:
# Cell 2 — Catalog summary statistics
#
# Full-catalog descriptive stats for VSSI, uncertainty, and key fields.
# Also: how many events have a named location vs. anonymous ('N/A' or blank)?

print('=== VSSI (Tg S) distribution — full catalog ===')
print(df['vssi_tg'].describe(percentiles=[.5,.75,.9,.95,.99]).round(3).to_string())

print(f'\nEvents with VSSI >= 5 Tg:  {(df.vssi_tg >= 5).sum()}')
print(f'Events with VSSI >= 10 Tg: {(df.vssi_tg >= 10).sum()}')
print(f'Events with VSSI >= 20 Tg: {(df.vssi_tg >= 20).sum()}')
print(f'Events with VSSI >= 50 Tg: {(df.vssi_tg >= 50).sum()}')

named = df['location'].notna() & (df['location'].str.strip() != 'N/A') & (df['location'].str.strip() != '')
print(f'\nNamed source volcano:  {named.sum()} / {len(df)} ({named.mean()*100:.1f}%)')
print(f'Anonymous events:      {(~named).sum()}')

print(f'\nTephra confirmed (Y):  {(df.tephra == "Y").sum()}')

print(f'\n=== Asymmetry distribution (DG/(DG+DA); 1=NH only, 0=SH only, 0.5=symmetric) ===')
print(df['asymmetry'].describe(percentiles=[.25,.5,.75]).round(3).to_string())
print(f'NH-dominant (>0.6):   {(df.asymmetry > 0.6).sum()}')
print(f'Symmetric (0.4-0.6):  {((df.asymmetry >= 0.4) & (df.asymmetry <= 0.6)).sum()}')
print(f'SH-dominant (<0.4):   {(df.asymmetry < 0.4).sum()}')

=== VSSI (Tg S) distribution — full catalog ===
count    256.000
mean       4.774
std        8.024
min        0.210
50%        1.900
75%        4.265
90%       12.140
95%       19.002
99%       39.739
max       59.420

Events with VSSI >= 5 Tg:  55
Events with VSSI >= 10 Tg: 33
Events with VSSI >= 20 Tg: 11
Events with VSSI >= 50 Tg: 2

Named source volcano:  42 / 256 (16.4%)
Anonymous events:      214

Tephra confirmed (Y):  21

=== Asymmetry distribution (DG/(DG+DA); 1=NH only, 0=SH only, 0.5=symmetric) ===
count    256.000
mean       0.659
std        0.406
min        0.000
25%        0.364
50%        0.852
75%        1.000
max        1.000
NH-dominant (>0.6):   165
Symmetric (0.4-0.6):  24
SH-dominant (<0.4):   67


In [3]:
# Cell 3 — Canonical event sanity check
#
# Verify the major historically documented eruptions are present
# with expected VSSI magnitudes. Also catches the 536 CE 'mystery eruption'
# which is well-established in the ice core record.

CANONICAL = [
    (536,  'Mystery eruption (536 CE) — famine of Justinian era'),
    (626,  'Unknown — early 7th c.'),
    (939,  'Eldgja (Iceland) — large flood basalt'),
    (1257, 'Samalas (Indonesia) — largest Holocene eruption'),
    (1452, 'Kuwae candidate — LIA onset context'),
    (1453, 'Kuwae candidate alt year'),
    (1600, 'Huaynaputina (Peru)'),
    (1815, 'Tambora (Indonesia) — Year Without a Summer'),
    (1883, 'Krakatoa (Indonesia)'),
]

print('Canonical event lookup (±2 year window):')
print(f'{"Year":>6}  {"Note":<45}  {"Found":>6}  {"VSSI (Tg)":>10}  {"Asymmetry":>10}  Location')
print('-'*110)
for yr, note in CANONICAL:
    window = df[(df.year_ad >= yr-2) & (df.year_ad <= yr+2)]
    if len(window) > 0:
        row = window.loc[window.vssi_tg.idxmax()]
        print(f'{yr:>6}  {note:<45}  {row.year_ad:>6}  {row.vssi_tg:>10.2f}  {row.asymmetry:>10.3f}  {str(row.location)[:40]}')
    else:
        print(f'{yr:>6}  {note:<45}  {"NOT FOUND":>6}')

Canonical event lookup (±2 year window):
  Year  Note                                            Found   VSSI (Tg)   Asymmetry  Location
--------------------------------------------------------------------------------------------------------------
   536  Mystery eruption (536 CE) — famine of Justinian era     536       18.81       1.000  nan
   626  Unknown — early 7th c.                            626       13.20       1.000  nan
   939  Eldgja (Iceland) — large flood basalt             939       16.23       1.000  Katla (Iceland)
  1257  Samalas (Indonesia) — largest Holocene eruption    1257       59.42       0.588  Samalas (Indonesia)
  1452  Kuwae candidate — LIA onset context              1453        9.97       0.829  nan
  1453  Kuwae candidate alt year                         1453        9.97       0.829  nan
  1600  Huaynaputina (Peru)                              1600       18.95       0.671  Huaynaputina (Peru)
  1815  Tambora (Indonesia) — Year Without a Summer      1815  

In [4]:
# Cell 4 — Events per century and VSSI distribution chart
#
# Left panel: events per century bar chart, colored by VSSI threshold.
# Shows quiet (Roman, Medieval) vs. active periods empirically.
# Right panel: VSSI histogram (log x-axis) — full catalog.

# Century bins: 1–100, 101–200, ..., 1901–2000 (using year_ad > 0)
# eVolv2k runs -429 to 1900 CE; we restrict to LMR window 1–1998 for relevance
df_lmr = df[(df.year_ad >= 1) & (df.year_ad <= 1998)].copy()
df_lmr['century'] = ((df_lmr.year_ad - 1) // 100) * 100 + 1  # 1, 101, 201, ...

century_counts = df_lmr.groupby('century').size()
century_major  = df_lmr[df_lmr.vssi_tg >= 10].groupby('century').size()

centuries = sorted(century_counts.index)
counts    = [century_counts.get(c, 0) for c in centuries]
majors    = [century_major.get(c, 0) for c in centuries]
minors    = [counts[i] - majors[i] for i in range(len(counts))]

xlabels = [f'{c}s' for c in centuries]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: events per century ---
ax = axes[0]
x = np.arange(len(centuries))
ax.bar(x, minors, color='steelblue', alpha=0.75, label='VSSI < 10 Tg')
ax.bar(x, majors, bottom=minors, color='firebrick', alpha=0.85, label='VSSI ≥ 10 Tg')
ax.set_xticks(x)
ax.set_xticklabels(xlabels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Event count')
ax.set_xlabel('Century CE')
ax.set_title('Events per century (LMR window 1–1998 CE)\nRed = major (VSSI ≥ 10 Tg)', fontsize=10)
ax.legend(fontsize=8)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Annotate notable quiet periods
for quiet_start, quiet_label in [(1, 'Roman quiet'), (901, 'Medieval quiet')]:
    idx = centuries.index(quiet_start) if quiet_start in centuries else None
    if idx is not None:
        ax.annotate(quiet_label, xy=(idx, counts[idx]+0.2),
                    fontsize=7, color='navy', ha='center')

# --- Right: VSSI distribution ---
ax2 = axes[1]
vssi_vals = df_lmr['vssi_tg'].dropna()
ax2.hist(vssi_vals, bins=np.logspace(np.log10(vssi_vals.min()+0.001),
                                      np.log10(vssi_vals.max()), 40),
         color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
ax2.set_xscale('log')
ax2.axvline(5,  color='orange', lw=1.2, ls='--', label='5 Tg (current API default)')
ax2.axvline(10, color='firebrick', lw=1.2, ls='--', label='10 Tg (major threshold)')
ax2.set_xlabel('VSSI (Tg S, log scale)')
ax2.set_ylabel('Event count')
ax2.set_title('VSSI distribution — LMR window (1–1998 CE)\nlog x-axis', fontsize=10)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / '07_events_per_century_vssi_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 07_events_per_century_vssi_dist.png')
print(f'Events in LMR window (1–1998 CE): {len(df_lmr)}')

Saved 07_events_per_century_vssi_dist.png
Events in LMR window (1–1998 CE): 211


In [6]:
# Cell 5 — Latitude and hemispheric asymmetry distribution
#
# Where are eruptions located? What fraction are genuinely NH, SH, or tropical?
# Asymmetry field: 1=NH only, 0=SH only, 0.5=both hemispheres equally.
# For API design: should we filter events by query-location hemisphere?

lat_vals  = df_lmr[df_lmr.vssi_tg >= 5]['lat'].dropna()
asym_vals = df_lmr[df_lmr.vssi_tg >= 5]['asymmetry'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: eruption latitude distribution ---
ax = axes[0]
# lat_vals = df_lmr['lat'].dropna()
lat_vals  = df_lmr[df_lmr.vssi_tg >= 5]['lat'].dropna() # filter to tg 5
ax.hist(lat_vals, bins=30, color='teal', alpha=0.8, edgecolor='white', linewidth=0.3)
ax.axvline(0,   color='black', lw=0.8, ls='-',  label='Equator')
ax.axvline(23,  color='gray',  lw=0.8, ls='--', label='Tropics (±23°)')
ax.axvline(-23, color='gray',  lw=0.8, ls='--')
ax.set_xlabel('Eruption latitude (°N)')
ax.set_ylabel('Event count')
ax.set_title('Eruption latitude distribution\n(LMR window 1–1998 CE)', fontsize=10)
ax.legend(fontsize=8)

nh = (lat_vals > 0).sum()
sh = (lat_vals < 0).sum()
trop = ((lat_vals >= -23) & (lat_vals <= 23)).sum()
print(f'NH eruptions (lat > 0):       {nh} ({nh/len(lat_vals)*100:.1f}%)')
print(f'SH eruptions (lat < 0):       {sh} ({sh/len(lat_vals)*100:.1f}%)')
print(f'Tropical (±23°):              {trop} ({trop/len(lat_vals)*100:.1f}%)')

# --- Right: asymmetry distribution ---
ax2 = axes[1]
# asym_vals = df_lmr['asymmetry'].dropna()
asym_vals = df_lmr[df_lmr.vssi_tg >= 5]['asymmetry'].dropna() # filter to tg 5
ax2.hist(asym_vals, bins=25, color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
ax2.axvline(0.4, color='orange',   lw=1, ls='--', label='SH-dominant threshold (< 0.4)')
ax2.axvline(0.6, color='firebrick',lw=1, ls='--', label='NH-dominant threshold (> 0.6)')
ax2.set_xlabel('Asymmetry (DG/(DG+DA); 0=SH only, 1=NH only)')
ax2.set_ylabel('Event count')
ax2.set_title('Hemispheric asymmetry distribution\n(LMR window 1–1998 CE)', fontsize=10)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / '07_latitude_asymmetry.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 07_latitude_asymmetry.png')

Saved 07_latitude_asymmetry.png


In [7]:
# Cell 6 — Empty-window analysis
#
# Slide windows of 50, 100, 200 years across the LMR period (1–1998 CE).
# For each window: is it empty? Does it contain ≥1 event at each VSSI threshold?
# This directly answers whether returning an event count is informative
# or whether most API queries will get null.
#
# Slide step = 10 years to get good coverage without oversampling.

VSSI_THRESHOLDS = [1, 5, 10, 20]  # Tg S
WINDOW_SIZES    = [50, 100, 200]   # years
STEP            = 10

results = []
for wsize in WINDOW_SIZES:
    starts = range(1, 1998 - wsize + 1, STEP)
    n_windows = len(list(starts))
    row = {'window_yr': wsize, 'n_windows': n_windows}
    for thr in VSSI_THRESHOLDS:
        count_with_event = 0
        for s in starts:
            e = s + wsize - 1
            window_events = df_lmr[(df_lmr.year_ad >= s) & (df_lmr.year_ad <= e) &
                                   (df_lmr.vssi_tg >= thr)]
            if len(window_events) > 0:
                count_with_event += 1
        pct = count_with_event / n_windows * 100
        row[f'pct_with_vssi_ge_{thr}'] = round(pct, 1)
        row[f'pct_empty_vssi_ge_{thr}'] = round(100 - pct, 1)
    results.append(row)

empty_df = pd.DataFrame(results)
print('=== Empty-window analysis ===')
print('% of query windows containing ≥1 event at each VSSI threshold:\n')
print(f'{"Window":>8}  {"≥1 Tg":>8}  {"≥5 Tg":>8}  {"≥10 Tg":>9}  {"≥20 Tg":>9}')
print('-'*52)
for _, r in empty_df.iterrows():
    print(f'{int(r.window_yr):>6}yr  '
          f'{r["pct_with_vssi_ge_1"]:>7.1f}%  '
          f'{r["pct_with_vssi_ge_5"]:>7.1f}%  '
          f'{r["pct_with_vssi_ge_10"]:>8.1f}%  '
          f'{r["pct_with_vssi_ge_20"]:>8.1f}%')
print()
print('(% empty = 100 - above)')
empty_df.to_csv(OUT / '07_empty_window_analysis.csv', index=False)
print('Saved 07_empty_window_analysis.csv')

=== Empty-window analysis ===
% of query windows containing ≥1 event at each VSSI threshold:

  Window     ≥1 Tg     ≥5 Tg     ≥10 Tg     ≥20 Tg
----------------------------------------------------
    50yr     95.4%     70.3%      50.8%      20.5%
   100yr     99.5%     96.8%      78.9%      36.8%
   200yr    100.0%    100.0%      97.2%      66.1%

(% empty = 100 - above)
Saved 07_empty_window_analysis.csv


In [8]:
# Cell 7 — Aggregation comparison across windows
#
# For each of the same sliding windows: compute three candidate aggregations:
#   (A) count of events with VSSI >= threshold
#   (B) sum of VSSI across all events in window
#   (C) years since most recent major event (VSSI >= 10) prior to window end
#
# Compare distributions to assess which provides more discriminating signal.
# Focus on 100-year windows as the representative query size.

WSIZE = 100
THR   = 5    # Tg — consistent with current API default
MAJOR = 10   # Tg — 'major' threshold for time-since-last
STEP  = 10

agg_rows = []
for s in range(1, 1998 - WSIZE + 1, STEP):
    e = s + WSIZE - 1
    window_df = df_lmr[(df_lmr.year_ad >= s) & (df_lmr.year_ad <= e)]
    events_above = window_df[window_df.vssi_tg >= THR]
    # (A) count
    count_a = len(events_above)
    # (B) sum VSSI
    sum_vssi = events_above['vssi_tg'].sum()
    # (C) years since last major event prior to or within window
    prior_major = df_lmr[(df_lmr.year_ad <= e) & (df_lmr.vssi_tg >= MAJOR)]
    if len(prior_major) > 0:
        last_yr = prior_major['year_ad'].max()
        time_since = e - last_yr
    else:
        time_since = None
    agg_rows.append({'window_start': s, 'window_end': e,
                     'count_vssi_ge5': count_a,
                     'sum_vssi': round(sum_vssi, 2),
                     'time_since_major': time_since})

agg_df = pd.DataFrame(agg_rows)

# Summary stats for each aggregation
print(f'=== Aggregation distributions (100-yr windows, VSSI threshold {THR} Tg) ===')
print(f'\n(A) Event count (VSSI ≥ {THR} Tg):')
print(agg_df['count_vssi_ge5'].describe(percentiles=[.5,.75,.9,.95]).round(2).to_string())
print(f'% windows with 0 events: {(agg_df.count_vssi_ge5 == 0).mean()*100:.1f}%')

print(f'\n(B) Sum VSSI (all events ≥ {THR} Tg, Tg S):')
print(agg_df['sum_vssi'].describe(percentiles=[.5,.75,.9,.95]).round(2).to_string())

print(f'\n(C) Years since last major event (VSSI ≥ {MAJOR} Tg):')
print(agg_df['time_since_major'].dropna().describe(percentiles=[.5,.75,.9,.95]).round(1).to_string())

# Correlation between (A) and (B)
corr_ab = agg_df[['count_vssi_ge5','sum_vssi']].corr().iloc[0,1]
print(f'\nCorrelation count vs. sum-VSSI: r = {corr_ab:.3f}')
print('(High r → count and sum-VSSI are largely interchangeable as summaries)')

agg_df.to_csv(OUT / '07_aggregation_comparison.csv', index=False)
print('Saved 07_aggregation_comparison.csv')

=== Aggregation distributions (100-yr windows, VSSI threshold 5 Tg) ===

(A) Event count (VSSI ≥ 5 Tg):
count    190.00
mean       2.15
std        1.24
min        0.00
50%        2.00
75%        3.00
90%        4.00
95%        5.00
max        5.00
% windows with 0 events: 3.2%

(B) Sum VSSI (all events ≥ 5 Tg, Tg S):
count    190.00
mean      33.99
std       27.24
min        0.00
50%       27.78
75%       47.07
90%       74.80
95%       89.63
max      119.83

(C) Years since last major event (VSSI ≥ 10 Tg):
count    190.0
mean      61.9
std       53.8
min        0.0
50%       47.0
75%       90.5
90%      139.2
95%      162.6
max      248.0

Correlation count vs. sum-VSSI: r = 0.868
(High r → count and sum-VSSI are largely interchangeable as summaries)
Saved 07_aggregation_comparison.csv


In [9]:
# Cell 8 — Aggregation visualization (100-yr sliding window)
#
# Plot count and sum-VSSI across the 2000-year window to show temporal structure.
# Highlights active periods (LIA onset, 6th-c. crisis) vs. quiet periods.

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

mid = (agg_df.window_start + agg_df.window_end) / 2

# Count
ax = axes[0]
ax.fill_between(mid, agg_df.count_vssi_ge5, alpha=0.6, color='steelblue')
ax.set_ylabel(f'Event count (VSSI ≥ {THR} Tg)')
ax.set_title(f'eVolv2k sliding 100-yr window aggregations (step={STEP} yr)', fontsize=11)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Sum VSSI
ax2 = axes[1]
ax2.fill_between(mid, agg_df.sum_vssi, alpha=0.6, color='firebrick')
ax2.set_ylabel('Sum VSSI (Tg S)')
ax2.set_xlabel('Window midpoint (CE)')

# Annotate notable periods on both axes
for a in axes:
    a.axvspan(100, 200,  alpha=0.08, color='green', label='Roman quiet (c.100–200 CE)')
    a.axvspan(950, 1100, alpha=0.08, color='blue',  label='Medieval quiet (c.950–1100 CE)')
    a.axvspan(500, 700,  alpha=0.08, color='orange',label='Late Antique active (c.500–700 CE)')

axes[0].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig(OUT / '07_window_aggregations_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 07_window_aggregations_timeseries.png')

Saved 07_window_aggregations_timeseries.png


In [10]:
# Cell 9 — Hemispheric filtering: does it matter for a query location?
#
# If a user queries northern Kaifeng (34°N) vs. southern Cape Town (−34°S),
# should the API filter events by hemispheric relevance (asymmetry)?
#
# Approach: for a set of VSSI thresholds, compare event counts when:
#   (all)  all events included
#   (nh)   NH-relevant only (asymmetry > 0.5, i.e. DG-dominant)
#   (sh)   SH-relevant only (asymmetry < 0.5, i.e. DA-dominant)
#
# Run on 100-yr windows; report median count under each filter.

WSIZE = 100
STEP  = 10
THR   = 5

def window_counts_by_filter(df_base, wsize, step, thr):
    rows = []
    for s in range(1, 1998 - wsize + 1, step):
        e = s + wsize - 1
        w = df_base[(df_base.year_ad >= s) & (df_base.year_ad <= e) & (df_base.vssi_tg >= thr)]
        rows.append({
            'all':  len(w),
            'nh':   len(w[w.asymmetry > 0.5]),
            'sh':   len(w[w.asymmetry < 0.5]),
            'symm': len(w[(w.asymmetry >= 0.4) & (w.asymmetry <= 0.6)]),
        })
    return pd.DataFrame(rows)

hemi_df = window_counts_by_filter(df_lmr, WSIZE, STEP, THR)

print(f'100-yr window event counts by hemispheric filter (VSSI ≥ {THR} Tg):')
print(f'{"Filter":<12}  {"Median":>8}  {"p75":>8}  {"p90":>8}  {"% empty":>10}')
print('-'*52)
for col, label in [('all','All events'), ('nh','NH-relevant'), ('sh','SH-relevant'), ('symm','Symmetric')]:
    s = hemi_df[col]
    print(f'{label:<12}  {s.median():>8.1f}  {s.quantile(.75):>8.1f}  '
          f'{s.quantile(.9):>8.1f}  {(s==0).mean()*100:>9.1f}%')

print()
reduction_nh = (1 - hemi_df['nh'].median() / hemi_df['all'].median()) * 100
reduction_sh = (1 - hemi_df['sh'].median() / hemi_df['all'].median()) * 100
print(f'Filtering to NH-relevant reduces median count by {reduction_nh:.0f}%')
print(f'Filtering to SH-relevant reduces median count by {reduction_sh:.0f}%')
print()
print('Interpretation: if reduction is small (< ~20%), hemispheric filtering adds')
print('little discriminating power. If large (> ~40%), filtering is worth exposing in API.')

hemi_df.to_csv(OUT / '07_hemispheric_filter_comparison.csv', index=False)
print('Saved 07_hemispheric_filter_comparison.csv')

100-yr window event counts by hemispheric filter (VSSI ≥ 5 Tg):
Filter          Median       p75       p90     % empty
----------------------------------------------------
All events         2.0       3.0       4.0        3.2%
NH-relevant        2.0       2.0       3.0       10.5%
SH-relevant        0.0       1.0       1.0       60.5%
Symmetric          1.0       1.0       2.0       45.8%

Filtering to NH-relevant reduces median count by 0%
Filtering to SH-relevant reduces median count by 100%

Interpretation: if reduction is small (< ~20%), hemispheric filtering adds
little discriminating power. If large (> ~40%), filtering is worth exposing in API.
Saved 07_hemispheric_filter_comparison.csv


In [11]:
# Cell 10 — Summary table and API design recommendations
#
# Consolidate key findings into a summary CSV and print recommendations.
# This becomes the primary input to the exploration_log.md Task 7 entries.

summary = {
    'catalog_n_total':          len(df),
    'catalog_n_lmr_window':     len(df_lmr),
    'year_range':               f"{df_lmr.year_ad.min()}–{df_lmr.year_ad.max()} CE",
    'vssi_median_tg':           round(df_lmr.vssi_tg.median(), 3),
    'vssi_p90_tg':              round(df_lmr.vssi_tg.quantile(.9), 3),
    'vssi_max_tg':              round(df_lmr.vssi_tg.max(), 2),
    'n_vssi_ge5':               int((df_lmr.vssi_tg >= 5).sum()),
    'n_vssi_ge10':              int((df_lmr.vssi_tg >= 10).sum()),
    'n_vssi_ge20':              int((df_lmr.vssi_tg >= 20).sum()),
    'pct_nh_dominant':          round((df_lmr.asymmetry > 0.6).mean() * 100, 1),
    'pct_sh_dominant':          round((df_lmr.asymmetry < 0.4).mean() * 100, 1),
    'corr_count_vs_sumvssi':    round(corr_ab, 3),
}

pd.DataFrame([summary]).T.rename(columns={0:'value'}).to_csv(
    OUT / '07_summary.csv'
)
print('=== Task 7 Summary ===')
for k, v in summary.items():
    print(f'  {k:<35} {v}')

print()
print('=== Provisional API design notes (to refine in exploration_log.md) ===')
print()
print('1. VSSI threshold: the 5 Tg default captures most named events but may be')
print('   too low to be interpretable (many small events). Revisit after empty-window')
print('   and aggregation results are in hand.')
print()
print('2. Aggregation: count and sum-VSSI are highly correlated (r shown above).')
print('   Sum-VSSI is marginally more informative (captures magnitude, not just presence).')
print('   Recommend: return count + sum-VSSI + time-since-last-major as three fields.')
print()
print('3. Hemispheric filtering: examine reduction % above. If < 20%, return all events')
print('   with asymmetry field and let users filter. If > 40%, offer query_lat parameter')
print('   for automatic hemispheric relevance filtering.')
print()
print('Saved 07_summary.csv')

=== Task 7 Summary ===
  catalog_n_total                     256
  catalog_n_lmr_window                211
  year_range                          4–1890 CE
  vssi_median_tg                      1.63
  vssi_p90_tg                         13.2
  vssi_max_tg                         59.42
  n_vssi_ge5                          41
  n_vssi_ge10                         26
  n_vssi_ge20                         9
  pct_nh_dominant                     62.6
  pct_sh_dominant                     28.0
  corr_count_vs_sumvssi               0.868

=== Provisional API design notes (to refine in exploration_log.md) ===

1. VSSI threshold: the 5 Tg default captures most named events but may be
   too low to be interpretable (many small events). Revisit after empty-window
   and aggregation results are in hand.

2. Aggregation: count and sum-VSSI are highly correlated (r shown above).
   Sum-VSSI is marginally more informative (captures magnitude, not just presence).
   Recommend: return count + sum-VSSI 